# OpenMontage — Full Pipeline Run (T4 x2)

End-to-end verification of the [OpenMontage](https://github.com/fafa433224/OpenMontage-beest) creative video pipeline on a free Kaggle T4 x2 instance:

1. **Clone + install** — repo, Python deps, Node 22 (Remotion + HyperFrames need it)
2. **Narration** — real speech via local Piper TTS (zero API keys)
3. **Open-Sora corpus** — real stock footage: stream-extract a slice of the LanguageBind/Open-Sora-Plan mixkit shard, CLIP-embed it into LanceDB
4. **Render** — the production harness `scripts/kaggle/render_voiceover_video.py`:
   GPU 0 = whisper transcription, GPU 1 = CLIP b-roll collection, dual-engine (Remotion body + HyperFrames kinetic intro/outro), SFX + BGM synthesis, seeded creative director (caption packs, per-cut transitions, per-scene backgrounds)
5. **QC** — duration/frame census, boundary frames, in-notebook playback

Optional: attach Kaggle secrets `LLM_BASE_URL` / `LLM_API_KEY` / `LLM_MODEL` (or any provider key) to get LLM creative direction instead of the seeded director.


In [ ]:
import subprocess, shutil, sys, os, json, time
from pathlib import Path

REPO = Path("/kaggle/temp/OpenMontage-beest")
INPUT = Path("/kaggle/temp/input")
OUT = Path("/kaggle/working/out")
OPEN_SORA_DIR = Path("/kaggle/temp/open-sora-dataset")
USE_OPEN_SORA = True   # real footage slice; falls back to synth b-roll on any failure

print("=== GPUs ===")
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
print("\n=== resources ===")
!df -h /kaggle/temp /kaggle/working | tail -3
!free -g | head -2


## 1 · Clone + install
Repo is public — plain clone, no token. GPU deps (torch, lancedb, faster-whisper, transformers for CLIP) + core deps + Piper TTS. Node 22 tarball goes in front of PATH: Kaggle's stock Node is too old for Remotion 4 / HyperFrames.


In [ ]:
import os, subprocess, time
t0 = time.time()

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/fafa433224/OpenMontage-beest.git", str(REPO)], check=True)
print("repo @", subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

# --- Node 22 (Remotion 4 / HyperFrames require >= 22) ---
NODE_VER = "22.14.0"
node22 = Path(f"/usr/local/node-v{NODE_VER}")
if not node22.exists():
    tar = f"node-v{NODE_VER}-linux-x64.tar.xz"
    subprocess.run(f"curl -fsSL https://nodejs.org/dist/v{NODE_VER}/{tar} -o /tmp/{tar}", shell=True, check=True)
    subprocess.run(f"tar -xJf /tmp/{tar} -C /usr/local", shell=True, check=True)
os.environ["PATH"] = f"{node22}/bin:" + os.environ.get("PATH", "")
print("node:", subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip())
print("npm: ", subprocess.run(["npm", "--version"], capture_output=True, text=True).stdout.strip())

# --- Python deps ---
r = subprocess.run(f"{sys.executable} -m pip install -q -r {REPO}/requirements.txt -r {REPO}/requirements-gpu.txt piper-tts 2>&1 | tail -5", shell=True, text=True)
print(r.stdout)
# av<14 pin can conflict with preinstalled av — enforce it explicitly
subprocess.run(f"{sys.executable} -m pip install -q 'av>=13.0,<14'", shell=True, check=True)

# --- Chrome shared libs for Remotion/HyperFrames headless shells ---
subprocess.run("apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y libnss3 libnspr4 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 libpango-1.0-0 libcairo2 >/dev/null 2>&1 || true", shell=True)
print(f"\nsetup done in {time.time()-t0:.0f}s")


In [ ]:
# Pre-download the Remotion headless browser while other things are cheap
r = subprocess.run("cd /kaggle/temp && npx --yes remotion browser ensure 2>&1 | tail -3",
                    shell=True, text=True)
print(r.stdout or "browser cached")


## 2 · Narration input (Piper TTS — local, free)
Same contract the repo's agent pipeline produces: `script.json` + `sections/*.mp3` + `voiceover.mp3`. ~150 words ≈ 60 s narration → ~62 s final video with the HyperFrames intro/outro.


In [ ]:
import json, subprocess, sys
sys.path.insert(0, str(REPO))

INPUT.mkdir(parents=True, exist_ok=True)
(INPUT / "sections").mkdir(exist_ok=True)

script = {
    "topic": "How black holes form",
    "title": "How Black Holes Form",
    "sections": [
        {"id": "s0", "heading": "A star runs out of fuel",
         "text": ("Every black hole begins as a massive star, at least twenty times heavier than our sun. For millions of years, the star burns hydrogen into helium, and that outward pressure holds the star up against its own crushing gravity.")},
        {"id": "s1", "heading": "Collapse in less than a second",
         "text": ("When the fuel finally runs out, the pressure vanishes and gravity wins instantly. The core collapses in under a second, and the outer layers blast away in a supernova explosion that can outshine a galaxy.")},
        {"id": "s2", "heading": "The point of no return",
         "text": ("What remains is packed into a region a few kilometers wide. Around it sits the event horizon, a boundary where escape velocity exceeds the speed of light. Nothing that crosses it can ever come back. Not light, not information, nothing.")},
        {"id": "s3", "heading": "Why they matter",
         "text": ("Studying black holes pushes physics to its limits, testing Einstein in the most extreme laboratory in the universe. Remember: every galaxy we have observed hosts a giant black hole at its center.")},
    ],
}
(INPUT / "script.json").write_text(json.dumps(script, indent=2))

# Pre-download the voice once, then synth each section
VOICE = "en_US-lessac-medium"
VOICES = Path("/kaggle/temp/piper-voices")
VOICES.mkdir(parents=True, exist_ok=True)
os.environ["PIPER_VOICE_DIR"] = str(VOICES)   # PiperTTS._voice_data_dirs() honors it
subprocess.run([sys.executable, "-m", "piper.download_voices", VOICE,
                "--data-dir", str(VOICES)], check=True)

from tools.audio.piper_tts import PiperTTS
tts = PiperTTS()

parts, total = [], 0.0
for s in script["sections"]:
    dest = INPUT / "sections" / f'{s["id"]}.mp3'
    r = tts.execute({"text": s["text"], "output_path": str(dest), "model": VOICE})
    assert r.success, f"piper failed: {r.error}"
    d = float(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                              "-of", "csv=p=0", str(dest)], capture_output=True, text=True).stdout)
    total += d; parts.append(str(dest)); print(f'{s["id"]}: {d:.2f}s')

listf = INPUT / "concat.txt"
listf.write_text("".join(f"file '{p}'\n" for p in parts))
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0",
                "-i", str(listf), "-c:a", "libmp3lame", "-b:a", "128k", "-ar", "22050",
                str(INPUT / "voiceover.mp3")], check=True)
vd = float(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                            "-of", "csv=p=0", str(INPUT / "voiceover.mp3")],
                           capture_output=True, text=True).stdout)
print(f"\nvoiceover.mp3: {vd:.2f}s — target is ~50-60s for the 1-minute test")


## 3 · Open-Sora-Dataset slice (real stock footage)
Full corpus is ~1 TB; we stream-extract just the head of the mixkit shard (26 GB compressed) — `curl | tar` pulls only what it reads — plus the LLaVA caption manifest, then CLIP-embed everything into LanceDB. The renderer's b-roll collector then serves **semantic search over real footage** instead of the procedural synth pool. Any failure here non-fatally falls back to synth b-roll, so the render always completes.


In [ ]:
import json, os, subprocess, time
note = "skipped"
if USE_OPEN_SORA:
    t0 = time.time()
    try:
        OPEN_SORA_DIR.mkdir(parents=True, exist_ok=True)
        videos = OPEN_SORA_DIR / "videos"
        videos.mkdir(exist_ok=True)

        # 3a. caption manifest (297 MB, HF Open-Sora-Plan layout: {path: caption})
        cap = OPEN_SORA_DIR / "llava_path_cap_64x512x512.json"
        if not cap.exists():
            subprocess.run(["curl", "-sL", "--retry", "3",
                "https://huggingface.co/datasets/LanguageBind/Open-Sora-Plan-v1.0.0/resolve/main/llava_path_cap_64x512x512.json",
                "-o", str(cap)], check=True, timeout=900)
        print(f"manifest: {cap.stat().st_size/2**20:.0f} MB")

        # 3b. stream-extract the head of the mixkit shard (bounded time)
        n_before = len(list(videos.rglob("*.mp4")))
        if n_before < 40:
            mixkit_url = ("https://huggingface.co/datasets/LanguageBind/"
                          "Open-Sora-Plan-v1.0.0/resolve/main/mixkit.tar.gz")
            inner = f"curl -sL --retry 3 {mixkit_url} | tar -xz -C {videos}"
            subprocess.run(f"timeout 240 bash -c {inner!r} || true",
                           shell=True, check=True)
        clips = list(videos.rglob("*.mp4"))
        print(f"extracted {len(clips)} real clips in {time.time()-t0:.0f}s")
        assert len(clips) >= 8, "too few clips extracted — check HF availability"

        # 3c. build the LanceDB + CLIP index (GPU 0; resumable)
        env = dict(os.environ, OPEN_SORA_DIR=str(OPEN_SORA_DIR))
        r = subprocess.run([sys.executable, str(REPO / "tools/video/open_sora_index.py"),
                            "build", "--dataset-dir", str(OPEN_SORA_DIR),
                            "--limit", "64"], env=env, text=True,
                           capture_output=True, timeout=1800)
        print(r.stdout[-1500:])
        assert r.returncode == 0, r.stderr[-800:]
        # Zero-row tripwire (belt + suspenders on top of the tool exit
        # code): a "successful" build that embedded nothing is the exact
        # silent-failure mode that once shipped a synth-fallback run.
        lines = r.stdout.splitlines()
        start = max(i for i, l in enumerate(lines) if l.strip() == "{")
        stats = json.loads("\n".join(lines[start:]))
        assert stats.get("total_rows", 0) > 0, f"index built 0 rows: {stats}"
        print(f"LanceDB index: {stats['total_rows']} clips embedded "
              f"({stats.get('error_count', 0)} errors)")
        # live semantic search proof — the exact path the renderer uses
        s = subprocess.run([sys.executable, str(REPO / "tools/video/open_sora_index.py"),
                           "search", "night sky full of stars", "--k", "3",
                           "--dataset-dir", str(OPEN_SORA_DIR)], env=env, text=True,
                          capture_output=True, timeout=300)
        print(s.stdout[-600:])
        assert "hits" in s.stdout and " 0 hits" not in s.stdout, s.stdout[-400:]
        note = "open_sora"
    except Exception as e:
        note = f"open-sora staging failed ({e}) — renderer will use synth b-roll"
        print(note)
else:
    print("USE_OPEN_SORA = False — synth b-roll pool")
print("BROLL SOURCE:", note)


## 4 · Optional LLM creative direction
With no secrets attached the **seeded creative director** runs solo (deterministic-but-varied caption packs, palettes, transitions). Attach `LLM_BASE_URL` + `LLM_API_KEY` (+ `LLM_MODEL`, `LLM_STYLE=openai|anthropic`) as Kaggle secrets and the LLM shapes the mood/palette/pacing instead. Both paths produce a complete video — the LLM only shifts creative choices.


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL", "LLM_STYLE"):
        try:
            v = sec.get_secret(name)
            if v: os.environ[name] = v; print(f"secret {name}: attached")
        except Exception:
            pass
except Exception:
    pass
print("LLM mode:", "configured" if os.environ.get("LLM_BASE_URL") or os.environ.get("OPENAI_API_KEY") else "seeded creative director (no keys)")


## 5 · Render — the production harness
`scripts/kaggle/render_voiceover_video.py`: measure → transcribe (GPU 0) → creative direction → props → npm ci → assets → Remotion body → HyperFrames intro/outro → ONE final video → QC. On T4 x2 the Open-Sora CLIP collector runs concurrently on GPU 1.


In [ ]:
import os, sys, subprocess, time
OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("OPEN_SORA_DIR", str(OPEN_SORA_DIR))
t0 = time.time()
cmd = [sys.executable, str(REPO / "scripts/kaggle/render_voiceover_video.py"),
       "blackholes", str(INPUT), str(REPO), str(OUT), "--concurrency", "4"]
r = subprocess.run(" ".join(cmd), shell=True, text=True)
print(f"\nrender exit={r.returncode} in {(time.time()-t0)/60:.1f} min")
assert r.returncode == 0, "render failed — see log above"


## 6 · QC + playback


In [ ]:
import json, subprocess
from IPython.display import Video, display, Image
from pathlib import Path

final = OUT / "blackholes.mp4"
if not final.exists():
    final = OUT / "blackholes-body.mp4"   # hyperframes fallback contract
print("final:", final.name, f"{final.stat().st_size/2**20:.1f} MB")

dur = float(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                             "-of", "csv=p=0", str(final)], capture_output=True, text=True).stdout)
frames = subprocess.run(["ffprobe", "-v", "error", "-count_frames", "-select_streams", "v:0",
                         "-show_entries", "stream=nb_read_frames", "-of", "csv=p=0", str(final)],
                        capture_output=True, text=True).stdout.strip()
print(f"duration {dur:.2f}s · frames {frames} · target ≈62s / ~1857 frames")

for f in ("blackholes-qc.json", "blackholes-timing.json"):
    p = OUT / f
    if p.exists():
        print(f"\n--- {f} ---")
        print(json.dumps(json.loads(p.read_text()), indent=1)[:1500])

# Real-footage contract: when Open-Sora staging succeeded the final video
# MUST have been rendered with real clips — a silent synth fallback here
# is a regression (this exact bug shipped as a green-looking v2 run).
t = OUT / "blackholes-timing.json"
if t.exists() and note == "open_sora":
    timing = json.loads(t.read_text())
    assert timing.get("broll_source") == "open_sora", (
        f"staging succeeded but renderer used {timing.get('broll_source')} "
        f"({timing.get('broll_note')}) — real-footage path is broken")
    print("\nREAL OPEN-SORA FOOTAGE CONFIRMED IN FINAL VIDEO: "
          f"{timing.get('broll_note')}")

census = OUT / "frames" / "census"
if census.is_dir():
    imgs = sorted(census.glob("*.png"))[:4]
    for im in imgs:
        print(im.name)
        display(Image(str(im), width=640))

display(Video(str(final), embed=True, width=800, mimetype="video/mp4"))


**Expected result:** one ~62 s, 1080p30 video — GSAP kinetic intro (5.4 s), four narrated sections with per-cut transitions (glitch / wipe / slide / rise / zoom-punch), per-scene animated backgrounds, real Open-Sora b-roll under hero cuts, SFX + BGM, kinetic outro (4.8 s). Everything produced with zero paid API keys on a free T4 x2 session.
